In [1]:
import numpy as np
import pandas as pd

In [2]:
proxy = pd.read_csv("../proxy/realized_cov_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()
ewma  = pd.read_csv("../models/ewma_cov_lambda094.csv", parse_dates=["Date"]).set_index("Date").sort_index()
dcc   = pd.read_csv("../results/dcc_cov_forecast_h21.csv", parse_dates=["Date"]).set_index("Date").sort_index()

In [3]:
print("Proxy shape:", proxy.shape)
print("EWMA shape:", ewma.shape)
print("DCC shape:", dcc.shape)

print("Proxy vs EWMA same columns:", list(proxy.columns) == list(ewma.columns))
print("Proxy vs DCC same columns:", list(proxy.columns) == list(dcc.columns))

common_ewma = proxy.index.intersection(ewma.index)
common_dcc  = proxy.index.intersection(dcc.index)

print("EWMA common dates:", len(common_ewma), common_ewma.min().date(), "->", common_ewma.max().date())
print("DCC common dates:", len(common_dcc), common_dcc.min().date(), "->", common_dcc.max().date())

Proxy shape: (2184, 64)
EWMA shape: (1255, 64)
DCC shape: (1255, 64)
Proxy vs EWMA same columns: True
Proxy vs DCC same columns: True
EWMA common dates: 1235 2021-01-04 -> 2025-12-02
DCC common dates: 1235 2021-01-04 -> 2025-12-02


In [4]:
def row_to_matrix(row, n_assets=8):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat

def make_spd(mat, eps=1e-10):
    mat = 0.5 * (mat + mat.T)
    mat = mat + eps * np.eye(mat.shape[0])
    return mat

def qlike_loss(proxy_mat, forecast_mat, eps=1e-10):
    S = make_spd(proxy_mat, eps=eps)
    H = make_spd(forecast_mat, eps=eps)

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)

def frobenius_loss(proxy_mat, forecast_mat):
    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):
    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "QLIKE": qlike_loss(S_mat, H_mat),
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean()
    })

    return losses, summary

In [5]:
ewma_losses, ewma_summary = evaluate_against_proxy(ewma, proxy, model_name="EWMA")
dcc_losses, dcc_summary = evaluate_against_proxy(dcc, proxy, model_name="DCC")

summary_table = pd.DataFrame([ewma_summary, dcc_summary]).reset_index(drop=True)
summary_table

,model,n_obs,start_date,end_date,avg_qlike,avg_frobenius
0,EWMA,1235,2021-01-04,2025-12-02,5.235591,0.000002
1,DCC,1235,2021-01-04,2025-12-02,4.488435,0.000017


In [6]:
loss_compare = pd.DataFrame({
    "EWMA_QLIKE": ewma_losses["QLIKE"],
    "DCC_QLIKE": dcc_losses["QLIKE"],
    "EWMA_FROB": ewma_losses["FROBENIUS"],
    "DCC_FROB": dcc_losses["FROBENIUS"],
})

loss_compare.describe()

print("DCC better on QLIKE share:",
      (dcc_losses["QLIKE"] < ewma_losses["QLIKE"]).mean())

print("DCC better on Frobenius share:",
      (dcc_losses["FROBENIUS"] < ewma_losses["FROBENIUS"]).mean())

DCC better on QLIKE share: 0.725506072874494
DCC better on Frobenius share: 0.3595141700404858


Varians and correlation performance


In [7]:
def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat, eps=1e-12):
    cov_mat = 0.5 * (cov_mat + cov_mat.T)

    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, eps))

    corr = cov_mat / np.outer(std, std)

    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr

In [8]:
def variance_mse_loss(proxy_mat, forecast_mat):

    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)

    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):

    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]

    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]

    return float(np.mean(diff ** 2))

In [9]:
def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets=8):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates].copy()
    proxy_df = proxy_df.loc[common_dates].copy()

    if list(forecast_df.columns) != list(proxy_df.columns):
        raise ValueError(f"Column mismatch between {model_name} and proxy.")

    records = []

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets=n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets=n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary

In [10]:
ewma_comp_losses, ewma_comp_summary = evaluate_components_against_proxy(
    ewma, proxy, model_name="EWMA"
)

dcc_comp_losses, dcc_comp_summary = evaluate_components_against_proxy(
    dcc, proxy, model_name="DCC"
)

component_summary_table = pd.DataFrame(
    [ewma_comp_summary, dcc_comp_summary]
).reset_index(drop=True)

component_summary_table

,model,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
0,EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
1,DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739


Høy/lav volatilitet 

In [12]:
proxy_test = proxy.loc[ewma_losses.index]

proxy_vol = proxy_test.apply(
    lambda row: np.trace(row_to_matrix(row)), axis=1
)

proxy_vol.name = "system_volatility"

proxy_vol.describe()

count    1235.000000
mean        0.001927
std         0.001188
min         0.000348
25%         0.001117
50%         0.001560
75%         0.002403
max         0.006489
Name: system_volatility, dtype: float64

In [13]:
low_threshold = proxy_vol.quantile(0.25)
high_threshold = proxy_vol.quantile(0.75)

print("Low volatility threshold:", low_threshold)
print("High volatility threshold:", high_threshold)

vol_regime = pd.Series(index=proxy_vol.index, dtype="object")

vol_regime[proxy_vol <= low_threshold] = "low"
vol_regime[proxy_vol >= high_threshold] = "high"
vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

vol_regime.value_counts()

Low volatility threshold: 0.001117003960638956
High volatility threshold: 0.0024032604891289543


mid     617
high    309
low     309
Name: count, dtype: int64

In [16]:
ewma_losses["regime"] = vol_regime.loc[ewma_losses.index]
dcc_losses["regime"] = vol_regime.loc[dcc_losses.index]

ewma_comp_losses["regime"] = vol_regime.loc[ewma_comp_losses.index]
dcc_comp_losses["regime"] = vol_regime.loc[dcc_comp_losses.index]

qlike_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["QLIKE"].mean(),
    "DCC": dcc_losses.groupby("regime")["QLIKE"].mean()
})

qlike_regime_table

frob_regime_table = pd.DataFrame({
    "EWMA": ewma_losses.groupby("regime")["FROBENIUS"].mean(),
    "DCC": dcc_losses.groupby("regime")["FROBENIUS"].mean()
})

frob_regime_table

var_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["VAR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["VAR_MSE"].mean()
})

var_regime_table

corr_regime_table = pd.DataFrame({
    "EWMA": ewma_comp_losses.groupby("regime")["CORR_MSE"].mean(),
    "DCC": dcc_comp_losses.groupby("regime")["CORR_MSE"].mean()
})

corr_regime_table

,EWMA,DCC
regime,,
high,0.101057,0.096769
low,0.073199,0.075213
mid,0.073534,0.065470


In [18]:
regime_summary = pd.concat(
    {
        "QLIKE": qlike_regime_table,
        "FROBENIUS": frob_regime_table,
        "VAR_MSE": var_regime_table,
        "CORR_MSE": corr_regime_table
    },
    axis=1
)

regime_summary

QLIKE               FROBENIUS                 VAR_MSE  \
            EWMA       DCC          EWMA       DCC          EWMA   
regime                                                             
high    7.515286  6.111136  3.213900e-06  0.000006  3.113609e-07   
low     4.218387  4.214621  6.027697e-07  0.000048  5.092294e-08   
mid     4.603322  3.812899  1.211043e-06  0.000008  1.171445e-07   

                      CORR_MSE            
                 DCC      EWMA       DCC  
regime                                    
high    6.091462e-07  0.101057  0.096769  
low     5.976040e-06  0.073199  0.075213  
mid     9.234530e-07  0.073534  0.065470

In [19]:
def system_volatility(df, n_assets=8):
    vol = df.apply(
        lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
        axis=1
    )
    return vol

realized_vol = system_volatility(proxy)
ewma_vol = system_volatility(ewma)
dcc_vol = system_volatility(dcc)

vol_df = pd.DataFrame({
    "Realized": realized_vol,
    "EWMA": ewma_vol,
    "DCC": dcc_vol
}).dropna()

vol_df.head()

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["Realized"],
    mode="lines",
    name="Realized volatility",
    line=dict(width=2)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["EWMA"],
    mode="lines",
    name="EWMA forecast",
    line=dict(width=1)
))

fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["DCC"],
    mode="lines",
    name="DCC forecast",
    line=dict(width=1)
))

fig.update_layout(
    title="System Volatility: Realized vs Forecast",
    xaxis_title="Date",
    yaxis_title="Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()